# T2a — Detección y tratamiento de nulos

**Autor:** Alejandro  
**Proyecto:** P1 — Análisis exploratorio del Titanic  
**Asignatura:** Python para IA — 2º DAM  

## Objetivo

Detectar, analizar e imputar o eliminar los valores nulos en el dataset Titanic siguiendo el contrato del equipo.

Las decisiones establecidas en el proyecto son:
- `age`: mediana
- `embarked` y `embark_town`: moda
- `deck`: se elimina la columna

El tratamiento se realizará de forma vectorizada, sin utilizar bucles `for` sobre las filas.

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset desde la carpeta Data del repositorio.
# Si no se encuentra (por ejemplo al abrir el notebook en Colab), se descarga
# de la misma fuente original, así que los datos son idénticos.
try:
    df = pd.read_csv("../Data/titanic.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")


# Mostramos las primeras filas
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
# Mostramos la cantidad de valores nulos por columna
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [3]:
# Hacemos una copia para aplicar el tratamiento
df_limpio = df.copy()

# Imputamos la mediana en la columna age
mediana_edad = df_limpio["age"].median()
df_limpio["age"] = df_limpio["age"].fillna(mediana_edad)

# Imputamos la moda en las columnas embarked y embark_town
moda_embarked = df_limpio["embarked"].mode()[0]
df_limpio["embarked"] = df_limpio["embarked"].fillna(moda_embarked)

moda_town = df_limpio["embark_town"].mode()[0]
df_limpio["embark_town"] = df_limpio["embark_town"].fillna(moda_town)

# Eliminamos la columna deck por exceso de nulos
df_limpio = df_limpio.drop(columns=["deck"])

## Justificación de cada decisión

| Columna | Nulos | Decisión | Por qué |
|---|---|---|---|
| `age` | 177 (19,9%) | Rellenar con la **mediana** (28 años) | Son demasiadas filas para borrarlas: perderíamos 1 de cada 5 pasajeros. Usamos la mediana y no la media (29,7) porque la edad tiene asimetría positiva y algunos valores altos (hasta 80 años) que tiran de la media hacia arriba; la mediana es más robusta frente a esos extremos. |
| `embarked` | 2 (0,2%) | Rellenar con la **moda** (`S`, Southampton) | Es una variable categórica, así que no tiene media ni mediana. Con solo 2 nulos, asignarles el puerto más frecuente apenas altera la distribución. |
| `embark_town` | 2 (0,2%) | Rellenar con la **moda** (Southampton) | Es la misma información que `embarked` en texto; se trata igual para que ambas columnas sigan siendo coherentes. |
| `deck` | 688 (77,2%) | **Eliminar la columna** | Falta en más de 3 de cada 4 pasajeros. Rellenarla sería inventarse la mayoría de los valores, y borrar esas filas dejaría el dataset en 203 pasajeros. |

**Limitación a tener en cuenta:** al imputar la edad, 177 pasajeros pasan a tener exactamente 28 años. Esto concentra artificialmente valores en la franja 18-59, algo que se tiene en cuenta al interpretar los resultados de edad en T5.

In [4]:
# Comprobamos que no queden valores nulos en el dataset
df_limpio.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

## Hallazgo

Se detectaron 869 valores nulos repartidos en `deck` (688), `age` (177) y `embarked`/`embark_town` (2 cada una).

El tratamiento deja el DataFrame con 0 nulos en todas sus variables y conservando íntegras las 891 filas originales.